In [1]:
import random
import time

import numpy as np
import pandas as pd

from experiment_classes import WrongIngTestSimple
from generate_recipe_task import *

from together import Together

recipe_df = pd.read_csv("csv/full_dataset.csv", index_col=0)
recipe_df.head()

,title,ingredients,directions,link,source,NER
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu..."
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom..."
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""cream cheese"", ""butter"", ""gar..."
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken"", ""chicken gravy"", ""cream of mushroo..."
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""peanut butter"", ""graham cracker crumbs"", ""bu..."


In [2]:
num_ings = [len(eval(row.loc['NER'])) for i, row in recipe_df.iterrows()]
num_steps = [len(eval(row.loc['directions'])) for i, row in recipe_df.iterrows()]
recipe_df['num_steps'] = num_steps
recipe_df['num_ings'] = num_ings

In [ ]:
ing_bins = [0, 8, 16, 32, 64, 128]
step_bins = [0, 8, 16, 32] #64, 128]

stepwise_bins_map = {}

for i in range(1, len(ing_bins)):
    for j in range(1, len(step_bins)):
        print(f'Current Bin: ing={ing_bins[i]}, step={step_bins[j]}')
        stepwise_bins_map[(ing_bins[i], step_bins[j])] = recipe_df[ing_bins[i] >= recipe_df['num_ings']][ing_bins[i-1] < recipe_df['num_ings']][step_bins[j] >= recipe_df['num_steps']][step_bins[j-1] < recipe_df['num_steps']]
        print(f'Recipes in current bin: {len(stepwise_bins_map[(ing_bins[i], step_bins[j])])}')
        print()

print(len(stepwise_bins_map))

In [11]:
for ing, step in stepwise_bins_map:
    cur = stepwise_bins_map[(ing, step)]
    cur.sample(min(50, len(cur))).to_csv(f'recipes_by_bin/s{step}i{ing}.csv')

# Transcript Processing

In [5]:
import random
import time

import numpy as np
import pandas as pd

from experiment_classes import WrongIngTestSimple
from generate_recipe_task import *

from together import Together
from tqdm import tqdm
import os

def process(files):
    for file in files:
        df = pd.read_csv(f'recipes_by_bin/{file}', index_col=0)

        print(f'Current Bin: {file}')
        print(f'Avg ingredients: {'%.3f'%np.mean(df['num_ings'])}')
        print(f'Avg steps: {'%.3f'%np.mean(df['num_steps'])}')
        print()
        print('Generating transcripts:')

        transcripts = []
        for i in tqdm(range(len(df))):
            row = df.iloc[i]
            if not isinstance(row['transcript'], float):
                transcripts.append(row['transcript'])
                continue
            rt = None

            try: rt = generate_ingredient_transcript(eval(row['ingredients']), eval(row['directions']), use_openai=True)
            except Exception as e: print(f'Exception {type(e)} for {row["title"]}')

            transcripts.append(rt)
        df['transcript'] = transcripts
        # print(df.columns)
        df.to_csv(f'recipes_by_bin/{file}')

        print()
        print('='*50)
        print()

In [32]:
files = os.listdir('recipes_by_bin')
# files.sort()
s8 = [file for file in files if file.startswith('s8')]
s8.sort(key=lambda s: len(s))
s16 = [file for file in files if file.startswith('s16')]
s16.sort(key=lambda s: len(s))
s32 = [file for file in files if file.startswith('s32')]
s32.sort(key=lambda s: len(s))

In [50]:
process(s8)

Current Bin: s8i8.csv
Avg ingredients: 5.660
Avg steps: 3.580

Generating transcripts:


100%|██████████| 50/50 [00:00<00:00, 63453.92it/s]




Current Bin: s8i32.csv
Avg ingredients: 19.800
Avg steps: 5.680

Generating transcripts:


100%|██████████| 50/50 [00:03<00:00, 14.86it/s]


Exception <class 'ValueError'> for Coconut Cake


Current Bin: s8i16.csv
Avg ingredients: 11.040
Avg steps: 4.320

Generating transcripts:


...

 38%|███▊      | 19/50 [00:05<00:08,  3.50it/s]

Exception <class 'ValueError'> for Murgh Makhanwala (Butter Chicken) 


100%|██████████| 50/50 [00:07<00:00,  6.58it/s]


Exception <class 'ValueError'> for Green Tea Crepes Filled With Azuki Cream And Pecans


Current Bin: s8i128.csv
Avg ingredients: 71.500
Avg steps: 4.333

Generating transcripts:


100%|██████████| 6/6 [00:00<00:00, 16131.94it/s]

In [49]:
process(s16)

Current Bin: s16i8.csv
Avg ingredients: 6.540
Avg steps: 11.000

Generating transcripts:


100%|██████████| 50/50 [00:18<00:00,  2.63it/s]


Exception <class 'ValueError'> for Vegetable Sandwiches


Current Bin: s16i32.csv
Avg ingredients: 19.780
Avg steps: 11.880

Generating transcripts:


 18%|█▊        | 9/50 [00:03<00:15,  2.67it/s]

Exception <class 'ValueError'> for Pumpkin Rum Pie


...

100%|██████████| 50/50 [00:42<00:00,  1.18it/s]




Current Bin: s16i16.csv
Avg ingredients: 11.060
Avg steps: 11.620

Generating transcripts:


100%|██████████| 50/50 [00:30<00:00,  1.63it/s]




Current Bin: s16i128.csv
Avg ingredients: nan
Avg steps: nan

Generating transcripts:


0it [00:00, ?it/s]

In [51]:
process(s32)
for file in files:
    df = pd.read_csv(f'recipes_by_bin/{file}', index_col=0)

Current Bin: s32i8.csv
Avg ingredients: 5.860
Avg steps: 20.280

Generating transcripts:


 12%|█▏        | 6/50 [00:00<00:06,  6.86it/s]

Exception <class 'ValueError'> for Chicken And Couscous Salad 


 36%|███▌      | 18/50 [00:19<00:37,  1.17s/it]

Exception <class 'ValueError'> for Buttermilk Biscuits


...

100%|██████████| 50/50 [01:10<00:00,  1.41s/it]


Exception <class 'ValueError'> for Baked Jerk Pork Loin Chops With Guava Sauce And Roasted Vegetables 


Current Bin: s32i16.csv
Avg ingredients: 12.840
Avg steps: 21.740

Generating transcripts:


100%|██████████| 50/50 [00:01<00:00, 27.60it/s]


Exception <class 'ValueError'> for Rum-Cured Salmon Summer Rolls


Current Bin: s32i128.csv
Avg ingredients: 78.000
Avg steps: 28.500

Generating transcripts:


100%|██████████| 2/2 [00:00<00:00, 9653.17it/s]

In [6]:
full = pd.read_csv('csv/stepwise_transcripts.csv', index_col=0)
for file in os.listdir('recipes_by_bin'):
    df = pd.read_csv(f'recipes_by_bin/{file}', index_col=0)
    if len(df) > 0: full = pd.concat((full, df))

full = full.loc[~full.index.duplicated(keep='first'), :]
full.to_csv(f'csv/stepwise_transcripts.csv')
full

,title,ingredients,directions,link,source,NER,num_steps,num_ings,transcript
2170889,Caribbean Curried Beef and Rice Supper,"[""1 lb. lean ground beef"", ""1 each: onion chop...","[""BROWN meat and onion; drain."", ""Add remainin...",www.kraftrecipes.com/recipes/caribbean-curried...,Recipes1M,"[""lean ground beef"", ""onion"", ""Rice"", ""raisins...",3,6,"[(['1 lb. lean ground beef', '1 each: onion ch..."
198164,Peanut Butter Fudge Pie,"[""2 chocolate graham cracker pie shells"", ""1 1...","[""Cream butter and sugar; add eggs, one at a t...",www.cookbooks.com/Recipe-Details.aspx?id=176737,Gathered,"[""graham cracker pie shells"", ""smooth peanut b...",6,7,"[(['1/2 lb. butter', '1 1/2 c. sugar', '6 eggs..."
1907723,Beer Pizza Crust (ABM),"[""1 cup beer"", ""1 teaspoon salt"", ""2 teaspoons...","[""Place all ingredients in machine in order re...",www.food.com/recipe/beer-pizza-crust-abm-508809,Recipes1M,"[""beer"", ""salt"", ""sugar"", ""bread flour"", ""basi...",7,6,"[(['1 cup beer', '1 teaspoon salt', '2 teaspoo..."
1499358,Wild Plum Jelly,"[""5 pounds wild plums, halved and pitted"", ""4 ...","[""In a stockpot, simmer plums and water until ...",www.tasteofhome.com/recipes/wild-plum-jelly/,Gathered,"[""wild plums"", ""water"", ""powdered fruit pectin...",3,4,"[(['5 pounds wild plums, halved and pitted', '..."
807694,Spiced Apple Cider,"[""2 qt. apple cider"", ""1 1/2 qt. cranberry coc...","[""Combine all ingredients in a large kettle."",...",www.cookbooks.com/Recipe-Details.aspx?id=166495,Gathered,"[""apple cider"", ""cranberry cocktail"", ""brown s...",2,6,"[(['2 qt. apple cider', '1 1/2 qt. cranberry c..."
...,...,...,...,...,...,...,...,...,...
1022104,Easy Loaded Baked Potatoes (4 Ways),"[""4 large baking potatoes (baked 1 hour at 375...","[""Preheat oven to 375\u00b0F."", ""Clean potatoe...",www.food.com/recipe/easy-loaded-baked-potatoes...,Gathered,"[""baking potatoes"", ""Variation"", ""alfredo sauc...",18,12,"[([], 'preheat', 'oven'), (['4 large baking po..."
1787943,"Roasted Tomato, Chickpea, and Swiss Chard Soup...","[""1 (28-ounce) can whole peeled tomatoes in th...","[""Heat the oven to 350 degrees F. Line a bakin...",www.chowhound.com/recipes/roasted-tomato-chick...,Recipes1M,"[""tomatoes"", ""olive oil"", ""Kosher salt"", ""Fres...",18,15,"[([], 'prepares', 'baking setup'), (['1 (28-ou..."
2138079,Prawn Karahi or Prawn Kadahi,"[""1 pound Prawns"", ""1 Tablespoon Cumin Seeds, ...","[""Remove the head, peel the prawns, and wash t...",tastykitchen.com/recipes/main-courses/prawn-ka...,Recipes1M,"[""Prawns"", ""Cumin Seeds"", ""Coriander Seeds"", ""...",20,16,"[(['1 pound Prawns'], 'wash', 'cleaned prawns'..."
1701304,B&B Chocolate Bouchee',"[""3-1/2 tbsp. cocoa powder"", ""1-1/2 cup all-pu...","[""Butter and flour 9 or 10-inch square pan."", ...",www.foodgeeks.com/recipes/1358,Recipes1M,"[""cocoa powder"", ""flour"", ""salt"", ""cinnamon"", ...",17,15,"[(['10 tbsp. butter', '1-1/2 cup all-purpose f..."


In [2]:
def valid_rows(df, params, actor_specs):
    rows = []
    for i, row in df.iterrows():
        try:
            transcript, used, unused = transcript_to_task(eval(row['transcript']), eval(row['ingredients']), actor_specs, params)
            # print(len(transcript.split('\n')), len(used), len(unused))
            if (len(transcript.split('\n')) >= params['transcript_length'] and
                    len(used) >= params['transcript_ings'] and
                    len(unused) >= params['unused_ings']):
                rows.append(i)
        except Exception as e: pass # print(f'Exception {type(e)} for {row["title"]}')
    return rows

In [10]:
params = {
    'transcript_ings': 16,
    'unused_ings': 4,
    'num_examples': 4,
    'transcript_length': 4,
}
actor_specs = ["Alice", "She", 1, 0.5, 0]

valid_16_4 = valid_rows(full, params, actor_specs)
print(len(valid_16_4))

55


In [11]:
params = {
    'transcript_ings': 4,
    'unused_ings': 16,
    'num_examples': 4,
    'transcript_length': 4,
}
actor_specs = ["Alice", "She", 1, 0.5, 0]

valid_4_16 = valid_rows(full, params, actor_specs)
print(len(valid_4_16))

58


# Ingredient Variables:
- ### \# of ingredients mentioned in transcript
- ### \# of ingredients tested
- ### \# of ingredients in recipe total

# Step Variables:
- ### \# of steps mentioned in transcript (transcript length)
- ### \# of steps in the recipe total

In [6]:
import pandas as pd
import os
from tqdm import tqdm
from generate_recipe_task import *
from experiment_classes import WrongIngTestSimple

stepwise_transcripts = pd.read_csv('csv/stepwise_transcripts.csv')

together_list = [
        "deepseek-ai/DeepSeek-R1",
        "OpenAI/gpt-oss-20B",
        "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
        "Qwen/Qwen3.5-397B-A17B",
        # "moonshotai/Kimi-K2-Instruct-0905",
    ]
openai_list = [
    'gpt-5',
    'gpt-5-mini',
]
testing_bins = [2,4,8,16]

Testing parameters:

4,2 4,4 4,8 4,16

2,4 4,4 8,4 16,4

dimensions:
- 6 models
- 8 parameter versions
- 50+ recipes

Organization:

per model results
- parameter csv
    - recipe transcript, used/unused, recipe examples, results

In [7]:
for model in openai_list:
    model_name = model.split('/')[-1]

    try: os.mkdir(f'results/{model_name}')
    except OSError as e: pass

    full = pd.read_csv('csv/stepwise_transcripts.csv', index_col=0)

    def test_on(axis):
        for bin in testing_bins:
            if os.path.exists(f'results/{model}/{axis}={bin}.csv'): continue

            params = {'transcript_ings': 4, 'unused_ings': 4, 'num_examples': 4, 'transcript_length': 4, axis: bin}
            metadata = {
                'id':[],
                'title': [],
                'transcript': [],
                'transcript_text': [],
                'used': [],
                'unused': [],
                'examples': [],
            }
            results = {
                'id':[],
                'outputs': [],
                'full_outputs': [],
                'correct': [],
                'overall_acc': []
            }

            actor_specs = ["Alice", "She", 1, 0.5, 0]

            valid = valid_rows(full, params, actor_specs)

            print(f'Testing {axis}={bin}:')
            for i in tqdm(range(min(len(valid), 50))):
                id = valid[i]
                row = full.loc[[id]]
                metadata['id'].append(id)
                metadata['title'].append(row['title'].item())
                transcript_text, used, unused = transcript_to_task(eval(row['transcript'].item()), eval(row['ingredients'].item()), actor_specs, params)
                metadata['transcript'].append(eval(row['transcript'].item()))
                metadata['transcript_text'].append(transcript_text)
                metadata['used'].append(used)
                metadata['unused'].append(unused)
                test = WrongIngTestSimple(transcript_text, used, unused, actor_specs, 0)
                metadata['examples'].append(test.perturb_recipe(max_examples=params['num_examples']))

                results['id'].append(id)
                outputs, full_outputs = test.run_test(client=OpenAI(), model=model, n=1)

                accuracies = test.get_results(match_exact=False)

                results['outputs'].append(outputs)
                results['full_outputs'].append(full_outputs)
                results['correct'].append(accuracies)
                results['overall_acc'].append(sum(accuracies)/len(accuracies))


            metadata_df = pd.DataFrame(metadata)
            metadata_df.to_csv(f'results/metadata/{axis}={bin}.csv', index=False)

            results_df = pd.DataFrame(results)
            results_df.to_csv(f'results/{model}/{axis}={bin}.csv', index=False)

    # vary transcript ingredients
    test_on('transcript_ings')
    test_on('unused_ings')



Testing unused_ings=8:


100%|██████████| 50/50 [1:14:17<00:00, 89.15s/it] 


Testing unused_ings=16:


100%|██████████| 50/50 [1:15:34<00:00, 90.70s/it] 


Testing transcript_ings=2:


...

100%|██████████| 50/50 [36:03<00:00, 43.27s/it]


Testing unused_ings=8:


100%|██████████| 50/50 [52:27<00:00, 62.94s/it]


Testing unused_ings=16:


100%|██████████| 50/50 [1:03:31<00:00, 76.23s/it] 


In [ ]:
for model in together_list:
    print(f'Testing model {model}:')

    model_name = model.split('/')[-1]

    try: os.mkdir(f'results/{model_name}')
    except OSError as e: pass

    full = pd.read_csv('csv/stepwise_transcripts.csv', index_col=0)

    def test_on(axis):
        for bin in testing_bins:
            if os.path.exists(f'results/{model_name}/{axis}={bin}.csv'): continue

            params = {'transcript_ings': 4, 'unused_ings': 4, 'num_examples': 4, 'transcript_length': 4, axis: bin}
            metadata = {
                'id':[],
                'title': [],
                'transcript': [],
                'transcript_text': [],
                'used': [],
                'unused': [],
                'examples': [],
            }
            results = {
                'id':[],
                'outputs': [],
                'full_outputs': [],
                'correct': [],
                'overall_acc': []
            }

            actor_specs = ["Alice", "She", 1, 0.5, 0]

            valid = valid_rows(full, params, actor_specs)

            print(f'Testing {axis}={bin}:')
            for i in tqdm(range(min(len(valid), 50))):
                id = valid[i]
                row = full.loc[[id]]
                metadata['id'].append(id)
                metadata['title'].append(row['title'].item())
                transcript_text, used, unused = transcript_to_task(eval(row['transcript'].item()), eval(row['ingredients'].item()), actor_specs, params)
                metadata['transcript'].append(eval(row['transcript'].item()))
                metadata['transcript_text'].append(transcript_text)
                metadata['used'].append(used)
                metadata['unused'].append(unused)
                test = WrongIngTestSimple(transcript_text, used, unused, actor_specs, 0)
                metadata['examples'].append(test.perturb_recipe(max_examples=params['num_examples']))

                results['id'].append(id)
                outputs, full_outputs = test.run_test(client=Together(), model=model, n=1)

                accuracies = test.get_results(match_exact=False)

                results['outputs'].append(outputs)
                results['full_outputs'].append(full_outputs)
                results['correct'].append(accuracies)
                results['overall_acc'].append(sum(accuracies)/len(accuracies))


            metadata_df = pd.DataFrame(metadata)
            metadata_df.to_csv(f'results/metadata/{axis}={bin}.csv', index=False)

            results_df = pd.DataFrame(results)
            results_df.to_csv(f'results/{model_name}/{axis}={bin}.csv', index=False)

    # vary transcript ingredients
    test_on('unused_ings')
    test_on('transcript_ings')



Testing model deepseek-ai/DeepSeek-R1:
Testing model OpenAI/gpt-oss-20B:
Testing model meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8:
Testing model Qwen/Qwen3.5-397B-A17B:
Testing unused_ings=16:


  0%|          | 0/50 [00:00<?, ?it/s]

In [23]:
client = Together()

## Uploads batch job file
file_resp = client.files.upload(
    file="results/DeepSeek-R1/transcript_ings=2.jsonl", purpose="batch-api", check=False
)

Uploading file transcript_ings=2.jsonl: 100%|██████████| 274k/274k [00:00<00:00, 458kB/s]


In [24]:
file_id = file_resp.id

batch = client.batches.create(
    input_file_id=file_id, endpoint="/v1/chat/completions"
)

print(batch.job.id)

24178496-8053-44ac-b3e6-d7eedfbebc00


In [26]:
batch_stat = client.batches.retrieve(batch.job.id)

print(batch_stat.status)

FAILED


# Old Test Code

In [5]:
def run_test(params, actor_specs, row):
    transcript, used, unused = transcript_to_task(eval(row['transcript']), eval(row['ingredients']), actor_specs, params)

    together_list = [
        "OpenAI/gpt-oss-20B",
        "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
        "Qwen/Qwen3-235B-A22B-Instruct-2507-tput",
        # "moonshotai/Kimi-K2-Instruct-0905",
        "deepseek-ai/DeepSeek-R1",
    ]
    openai_list = [
        'gpt-5',
        'gpt-5-mini',
    ]

    results = {
        'model': [],
        'outputs': [],
        'full_outputs': [],
        'accuracy per ingredient': [],
        'overall accuracy': [],
    }

    test = WrongIngTestSimple(transcript, used, unused, actor_specs, 0)
    print(test.perturb_recipe(False))

    for model in together_list:
        print(f'Testing model {model}: ')
        outputs, full_outputs = test.run_test(client=Together(), model=model)
        accuracies = test.get_results()
        results['model'].append(model)
        results['outputs'].append(outputs)
        results['full_outputs'].append(full_outputs)
        results['accuracy per ingredient'].append(accuracies)
        results['overall accuracy'].append(sum(accuracies)/len(accuracies))

    for model in openai_list:
        print(f'Testing model {model}: ')
        outputs, full_outputs = test.run_test(client=OpenAI(), model=model)
        accuracies = test.get_results()
        results['model'].append(model)
        results['outputs'].append(outputs)
        results['full_outputs'].append(full_outputs)
        results['accuracy per ingredient'].append(accuracies)
        results['overall accuracy'].append(sum(accuracies)/len(accuracies))

    return transcript, used, unused, results



In [6]:
import os

params = {
    'transcript_ings': 4,
    'unused_ings': 4,
    'num_examples': 4,
    'transcript_length': 5,
}
actor_specs = ["Alice", "She", 1, 0.5, 0]

# print(len(eval(row['transcript'])))
# for ings, action, product in eval(row['transcript']):
#     print(ings)

# valid_rows = [16, 22, 25, 27, 28, 29, 32, 33, 35, 37, 38]
valid_rows = [29, 32, 33, 35, 37, 38]

for i in valid_rows:
    row = stepwise_transcripts.iloc[i]

    try: os.mkdir(f'results/{row.iloc[0]}')
    except OSError as e: pass

    r1 = list(stepwise_transcripts.columns)
    r1[0] = 'index'
    r2 = list(row)
    metadata = pd.DataFrame(r2, r1)
    metadata.to_csv(f'results/{row.iloc[0]}/metadata.csv', header=False)

    inputs = {
        'transcript_ings': [],
        'unused_ings': [],
        'transcript_length': [],
        'transcript': [],
        'used_list':[],
        'unused_list':[],
    }

    for t in [2,4,8]:
        for u in [2,4,8]:
            params = {
                'transcript_ings': t,
                'unused_ings': u,
                'transcript_length': 5,
            }
            transcript, used, unused, results = run_test(params, actor_specs, row)

            inputs['transcript_ings'].append(t)
            inputs['unused_ings'].append(u)
            inputs['transcript_length'].append(5)
            inputs['transcript'].append(transcript)
            inputs['used_list'].append(used)
            inputs['unused_list'].append(unused)

            outputs = pd.DataFrame(results)
            outputs.to_csv(f'results/{row.iloc[0]}/outputs-t{t}u{u}.csv', index=False, header=True)

    inputs = pd.DataFrame(inputs)
    print(inputs.columns)
    inputs.to_csv(f'results/{row.iloc[0]}/inputs.csv', index=False, header=True)

Recipe 2
 ['1/3 cups Canola Or Vegetable Oil', '1-1/2 cup Whole Wheat Pastry Flour', '1/2 teaspoons Ground Nutmeg']

Recipe 3
 ['1/3 cups Canola Or Vegetable Oil', '1-1/2 cup Whole Wheat Pastry Flour', '1 teaspoon Ground Cinnamon']


Testing model OpenAI/gpt-oss-20B: 
Running recipe with '1 teaspoon Ground Cinnamon', spec level 0.
[*****]
Running recipe with '1/2 teaspoons Ground Nutmeg', spec level 0.
[*****]
Testing model meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8: 
Running recipe with '1 teaspoon Ground Cinnamon', spec level 0.
[*****]
Running recipe with '1/2 teaspoons Ground Nutmeg', spec level 0.
[*****]
Testing model Qwen/Qwen3-235B-A22B-Instruct-2507-tput: 
Running recipe with '1 teaspoon Ground Cinnamon', spec level 0.
[*****]
Running recipe with '1/2 teaspoons Ground Nutmeg', spec level 0.
[*****]
Testing model deepseek-ai/DeepSeek-R1: 
Running recipe with '1 teaspoon Ground Cinnamon', spec level 0.
[*****]
Running recipe with '1/2 teaspoons Ground Nutmeg', spec level 

KeyboardInterrupt: 

# TODO:
- test one var at a time, n=1 trials
- increase reasoning tokens for together models
- split unused_ings/# recipes